In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=4, d_ff=512, dropout=0.1):
        super().__init__()

        # Normalization and creating heads
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

        # Drops 10% of the residual connection
        self.residual_dropout = nn.Dropout(dropout)

    def forward(self, h, causal_mask, padding_mask):
        # Attention sublayer
        normalized = self.norm1(h)
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=causal_mask,
            key_padding_mask=padding_mask,
            need_weights=False,
        )
        h = h + self.residual_dropout(attended)

        # Feed-forward sublayer
        h = h + self.residual_dropout(
            self.feed_forward(self.norm2(h))
        )
        return h

In [8]:
class CausalProteinTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=23,
        max_length=129,
        d_model=128,
        n_heads=4,
        n_layers=4,
        d_ff=512,
        dropout=0.1,
        pad_id=0,
    ):
        super().__init__()
        self.pad_id = pad_id

        self.token_embedding = nn.Embedding(
            vocab_size, d_model, padding_idx=pad_id
        )
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.embedding_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, 3) # Down to the 3 families


    def forward(self, x):
        _, T = x.shape

        positions = torch.arange(T, device=x.device)
        h = self.token_embedding(x) + self.position_embedding(positions)
        h = self.embedding_dropout(h)

        causal_mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        padding_mask = x.eq(self.pad_id)

        for block in self.blocks:
            h = block(h, causal_mask, padding_mask)

        #return self.output(self.final_norm(h))
        return F.softmax(self.output(self.final_norm(h)), dim=2) # F.softmax(x, dim=2) # softmax to be performed on the third dimension


In [9]:
model = CausalProteinTransformer(
    vocab_size=23,
    max_length=129,
    pad_id=0,
)  # Use the same class and arguments as during training
model.load_state_dict(torch.load("classifierModel.pt", weights_only=True))
model.eval()

CausalProteinTransformer(
  (token_embedding): Embedding(23, 128, padding_idx=0)
  (position_embedding): Embedding(129, 128)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (feed_forward): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
      )
      (residual_dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (output): Linear(in_features=128, out_features=3, bias=T